<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3-Nano-Sim-Bimanual with SGLang

Replay the published `agibot_mixed_028` example of `nvidia/Cosmos3-Nano-Sim-Bimanual` through a persistent SGLang server: a mixed-domain sequence in which AgiBot manipulation rows alternate with camera-pose rows, each row tagged with its embodiment. Its 900 normalized unified 59D actions produce **901 frames at 30 FPS** with seed **2026** from the published initial image, a 640×360 frame. The server pads the frame into the trained 832×480 canvas for VAE encoding and drops the padded latents, so the output keeps the image's size on the 16-pixel latent grid, 640×352. The notebook uses an HTTP client and local CPU helpers; the SGLang server owns the model and checkpoint configuration.

The model predicts the visual result of supplied actions. It does not choose robot actions or feed generated observations back into a policy.

| Component | Version |
| --- | --- |
| SGLang | The published [`lmsysorg/sglang:dev`](https://hub.docker.com/r/lmsysorg/sglang/tags) image, which carries the Cosmos Dreams pipeline from [sgl-project/sglang#39475](https://github.com/sgl-project/sglang/pull/39475); the tag moves, so pull it before starting the server |
| Model and examples | The Hugging Face revisions selected by `COSMOS3_MODEL_REVISION` and `COSMOS3_EXAMPLE_REVISION`, by default the repository's current default revision |

## 1. Model and runtime contract

The server loads the exported `nvidia/Cosmos3-Nano-Sim-Bimanual` artifact. Its `model_index.json` must select `Cosmos3NanoSimBimanualPipeline`.

This is an **autoregressive, self-forcing DMD-distilled, 480p** checkpoint. SGLang reads the causal rollout and sampling configuration from the export: the chunk size in latent frames, the fixed-step denoising schedule, and the history mode. The start-up log prints the resolved values. Keep the exported `transformer/config.json` and `checkpoint.json` intact.

The `conditioning` contract (schema 5) must be preserved. It defines the 59-value `unified_v1` row that every embodiment shares, the embodiment table (`agibotworld` is domain 15, `camera_pose` is domain 2), the `backward_chunk_anchored_16f` pose convention, and the `global_asinh_unified_v1` normalizer that the published sidecars already apply. The server pads each row to 64 values and routes it to the domain named for that row. Chunk and window sizes count latent frames; one latent frame corresponds to four video frames. The training loader sent prompts as plain text, so the server tokenizes the prompt verbatim unless a request asks for the JSON caption.

The server runs in the published SGLang image, `lmsysorg/sglang:dev`. Pull it on the server host before each start:

```bash
docker pull lmsysorg/sglang:dev
```

The `dev` tag moves with SGLang's main branch. This notebook was validated against the image built from SGLang commit [`ab580b2e`](https://github.com/sgl-project/sglang/commit/ab580b2e060059c0f8387b598a9f648dcec13f93), published as the fixed tag `lmsysorg/sglang:nightly-dev-20261020-ab580b2e`; use that tag in place of `dev` to reproduce the validated run.

The notebook client needs FFmpeg with `ffprobe` for output verification.

## 2. Start the server

`SGLANG_HF_CACHE` is a host directory for the Hugging Face cache, and `HF_TOKEN` needs read access to the checkpoint. The container runs as root by default, so the setup lines create that directory world-writable; alternatively, start the container with your host user ID. `COSMOS3_MODEL_REVISION` selects the checkpoint revision; leave it unset for the repository's default. Run this in a terminal and leave it running while using the notebook:

```bash
export SGLANG_HF_CACHE="${SGLANG_HF_CACHE:-$HOME/.cache/sglang-huggingface}"
mkdir -p "$SGLANG_HF_CACHE"
chmod 777 "$SGLANG_HF_CACHE"
docker run --init --runtime nvidia --gpus all \
  -v $SGLANG_HF_CACHE:/root/.cache/huggingface \
  -e HF_TOKEN=$HF_TOKEN \
  -p 30000:30000 \
  --ipc=host \
  lmsysorg/sglang:dev \
  sglang serve \
  --model-path nvidia/Cosmos3-Nano-Sim-Bimanual \
  --revision "${COSMOS3_MODEL_REVISION:-main}" \
  --host 0.0.0.0 --port 30000
```

The first start downloads the checkpoint into the mounted cache. The HTTP port opens only after the weights are loaded and the warmup requests finish, so wait for the `Uvicorn running` line. The start-up log also prints the resolved checkpoint, the denoising steps per chunk, and the history mode. The notebook uploads the initial image and sends the actions inline, so the server needs no access to the client's files. For a remote notebook, set `SGLANG_BASE_URL`, or forward the port with `ssh -L 30000:127.0.0.1:30000 <server-host>`.

## 3. Set up the client

Install these packages in the notebook kernel environment, and authenticate to Hugging Face with read access to `nvidia/Cosmos3-Nano-Sim-Bimanual`; run `hf auth login` or set `HF_TOKEN`:

```bash
pip install numpy Pillow httpx matplotlib ipython huggingface_hub orjson
```

`orjson` is optional: it writes the action payload compactly, and without it the helper falls back to the standard library with the same result.

Install FFmpeg with `ffprobe` on the client as well. The local [`bimanual_helpers.py`](bimanual_helpers.py) prepares the image and actions with NumPy and Pillow, then submits HTTP requests. The client does not import an inference runtime or download checkpoint weights; SGLang validates the exported model and conditioning contract on the server.

| Variable | Default | Purpose |
| --- | --- | --- |
| `SGLANG_BASE_URL` | `http://127.0.0.1:30000` | Server address |
| `SGLANG_API_KEY` | unset | Bearer token, if the server was started with `--api-key` |
| `BIMANUAL_OUTPUT_DIR` | `outputs/sglang/bimanual` under this notebook's folder | Output folder |
| `COSMOS3_MODEL_REVISION` | `main` | Checkpoint revision; set it to the same value the server was started with |
| `COSMOS3_EXAMPLE_REVISION` | `main` | Revision of the published manifests, images, and action sidecars |
| `COSMOS3_BIMANUAL_MANIFEST` | `examples/inference/sim_bimanual_mixed_action_hf.jsonl` | Manifest to read the example row from; `examples/inference/sim_bimanual_unifiedv1_hf.jsonl` holds the single-domain examples |
| `COSMOS3_BIMANUAL_ACTION` | `agibot_mixed_028_normalized_mixed_action.json` | Basename of the row's action sidecar, which selects the example |

Run the notebook from the cookbook root (`.agents/skills`), the simulation folder, or this `bimanual` folder.

In [ ]:
from pathlib import Path
from fractions import Fraction
from urllib.parse import unquote
import json
import math
import os
import shutil
import subprocess
import sys
import time

import httpx
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Video, display

start = Path.cwd().resolve()
SIMULATION_ROOT = next(
    (
        candidate
        for root in (start, *start.parents)
        for candidate in (root, root / "cookbooks" / "cosmos3" / "simulation", root / ".agents" / "skills" / "cookbooks" / "cosmos3" / "simulation")
        if (candidate / "bimanual" / "bimanual_helpers.py").is_file()
    ),
    None,
)
if SIMULATION_ROOT is None:
    raise FileNotFoundError("Run inside .agents/skills, which contains cookbooks/cosmos3/simulation/bimanual/bimanual_helpers.py.")
BIMANUAL_ROOT = SIMULATION_ROOT / "bimanual"
if str(BIMANUAL_ROOT) not in sys.path:
    sys.path.insert(0, str(BIMANUAL_ROOT))

MODEL_ID = "nvidia/Cosmos3-Nano-Sim-Bimanual"
MODEL_REVISION = os.environ.get("COSMOS3_MODEL_REVISION", "main")
EXAMPLE_REVISION = os.environ.get("COSMOS3_EXAMPLE_REVISION", "main")
MANIFEST_NAME = os.environ.get("COSMOS3_BIMANUAL_MANIFEST", "examples/inference/sim_bimanual_mixed_action_hf.jsonl")
ACTION_BASENAME = os.environ.get("COSMOS3_BIMANUAL_ACTION", "agibot_mixed_028_normalized_mixed_action.json")
SGLANG_BASE_URL = os.environ.get("SGLANG_BASE_URL", "http://127.0.0.1:30000").rstrip("/")
SGLANG_API_KEY = os.environ.get("SGLANG_API_KEY")
HEADERS = {"Authorization": f"Bearer {SGLANG_API_KEY}"} if SGLANG_API_KEY else {}
HTTP_TIMEOUT = 3600.0
OUTPUT_DIR = Path(
    os.environ.get("BIMANUAL_OUTPUT_DIR", BIMANUAL_ROOT / "outputs" / "sglang" / "bimanual")
).expanduser().resolve()

assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "Install FFmpeg and ffprobe."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Server:", SGLANG_BASE_URL, "(with API key)" if SGLANG_API_KEY else "")
print("Model:", MODEL_ID, "@", MODEL_REVISION)
print("Example:", MANIFEST_NAME, "/", ACTION_BASENAME, "@", EXAMPLE_REVISION)
print("Outputs:", OUTPUT_DIR)

### Verify `/v1/models`

Check that the server is ready and advertises the Bimanual model, then display its model list.

In [ ]:
with httpx.Client(base_url=SGLANG_BASE_URL, headers=HEADERS, timeout=30) as client:
    models_response = client.get("/v1/models")
    models_response.raise_for_status()
    models_payload = models_response.json()
model_ids = [item["id"] for item in models_payload.get("data", []) if item.get("id")]
if not model_ids:
    raise RuntimeError(f"The server returned no model IDs: {models_payload}")
if MODEL_ID not in model_ids:
    raise RuntimeError(f"Expected {MODEL_ID}; the server advertises {model_ids}")
ACTIVE_MODEL = MODEL_ID
print("active model:", ACTIVE_MODEL)
print(json.dumps(models_payload, indent=2))

In [ ]:
from huggingface_hub import hf_hub_download

from bimanual_helpers import domain_runs, prepare_bimanual_input, submit_bimanual

# Preparation is local CPU work. The server applies its checkpoint contract.

## 4. Download the published example

The manifest, by default `examples/inference/sim_bimanual_mixed_action_hf.jsonl`, comes from the model repository at `COSMOS3_EXAMPLE_REVISION`. Each row names its initial image and action sidecar by Hugging Face URL; the next cell selects the row whose sidecar is `COSMOS3_BIMANUAL_ACTION`, by default `agibot_mixed_028`, downloads both files into the Hugging Face cache, and keeps the row's frame count, FPS, seed, resolution, action width, and prompt format for the request. The action values are already normalized and validity-masked; the server pads the 59 input channels to the model's 64.

To use your own recording, set these before starting the notebook; the published row still supplies the frame count, FPS, seed, and prompt format:

| Variable | Replaces |
| --- | --- |
| `ACTION_PATH` | The sidecar: a JSON object with `action_space: "normalized_unified_v1"`, a `900 × 59` `action` array, and `domain_names`, one per row or one for the whole clip. |
| `VISION_PATH` | The initial observation: an image, or a video whose first frame is used. |
| `HUMANOID_PROMPT` | The task description, sent as plain text. |

In [ ]:
manifest_path = Path(hf_hub_download(repo_id=MODEL_ID, filename=MANIFEST_NAME, revision=EXAMPLE_REVISION))
rows = [json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
row = next(item for item in rows if item["action_path"].endswith("/" + ACTION_BASENAME))


def download_hosted_asset(url: str) -> Path:
    prefix = f"https://huggingface.co/{MODEL_ID}/resolve/main/"
    if not url.startswith(prefix):
        raise ValueError(f"Expected a published {prefix} URL, got {url!r}")
    return Path(hf_hub_download(repo_id=MODEL_ID, filename=unquote(url.removeprefix(prefix)), revision=EXAMPLE_REVISION))


VISION_PATH = Path(os.environ.get("VISION_PATH") or download_hosted_asset(row["vision_path"])).expanduser().resolve()
ACTION_PATH = Path(os.environ.get("ACTION_PATH") or download_hosted_asset(row["action_path"])).expanduser().resolve()
PROMPT = os.environ.get("HUMANOID_PROMPT", row["prompt"])
record = {key: row[key] for key in ("num_frames", "fps", "seed", "resolution", "raw_action_dim", "format_prompt_as_json") if key in row}
record.update(prompt=PROMPT, vision_path=str(VISION_PATH), action_path=str(ACTION_PATH))

for path in (ACTION_PATH, VISION_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Missing input; check the download or your override: {path}")
print(f"{row['name']}: {record['num_frames']} frames at {record['fps']} FPS, seed {record['seed']}")
print("Actions:", ACTION_PATH)
print("Initial image:", VISION_PATH)

### Action structure

The sidecar has `action_space: "normalized_unified_v1"`, a `900×59` action array, and one domain name per row: `agibotworld` for manipulation rows and `camera_pose` for camera-motion rows, which the checkpoint maps to domain IDs 15 and 2. The rows alternate between the two domains in runs of varying length, and the next cell prints that run sequence as `(domain, row count)` pairs. At the release revision of the model repository the layout is a short camera lead-in, two long blocks that switch at rows 300 and 600, and several brief alternations over the final six seconds; a sidecar with a different layout needs no notebook changes, because the domain names travel with the request. The initial image plus 900 action rows produce 901 frames. The 59 columns are grouped as follows (Python slices exclude the end index):

| Columns | Contents |
| --- | --- |
| `0:9` | Ego/camera translation and rotation |
| `9:18` | Right-wrist translation and rotation |
| `18:33` | Right fingertips |
| `33` | Right-gripper width |
| `34:43` | Left-wrist translation and rotation |
| `43:58` | Left fingertips |
| `58` | Left-gripper width |

Poses use the checkpoint's `backward_chunk_anchored_16f` convention and six rotation values (the first two matrix columns). Values are already globally asinh-normalized; unavailable channels are masked to zero after normalization. Do not normalize again: the request sends `action_normalization="none"`, the server pads 59 channels to 64, and camera rows may only carry values in columns `0:9`.

In [ ]:
sidecar = json.loads(ACTION_PATH.read_text(encoding="utf-8"))
assert sidecar["action_space"] == "normalized_unified_v1", sidecar.get("action_space")
actions = sidecar["action"]
domain_names = sidecar["domain_names"]
if isinstance(domain_names, str):
    domain_names = [domain_names] * len(actions)
assert isinstance(actions, list) and len(actions) == record["num_frames"] - 1, "expected num_frames - 1 action rows"
assert len(domain_names) == len(actions), "one domain name is required per action row"
for row_index, values in enumerate(actions):
    assert isinstance(values, list) and len(values) == 59, f"row {row_index}: expected 59 values"
    for value in values:
        assert isinstance(value, (int, float)) and not isinstance(value, bool), f"row {row_index}: non-numeric value"
        assert math.isfinite(value), f"row {row_index}: non-finite value"
print(f"Validated {len(actions)}×{len(actions[0])} finite actions from {ACTION_PATH}")
print("Domain sequence:", [(name, end - start) for name, start, end in domain_runs(domain_names)])

### Prepare the input

The initial image provides the starting observation. The 900 action rows, each routed to the domain named for it, drive the following frames, producing a 901-frame video. The helper keeps the rows as they are and sends one embodiment name per row; the server applies no further normalization. The cell also prints the domain runs as `(domain, first_row, end_row)` with the end excluded, which locate every switch in the output. It also reports the canvas the server encodes and the expected output size: for the published 640×360 image, 832×480 and 640×352.

In [ ]:
humanoid_input = prepare_bimanual_input(record, base_dir=ACTION_PATH.parent)
assert humanoid_input.action.shape == (record["num_frames"] - 1, 59)
assert humanoid_input.extra_args["action_normalization"] == "none"
assert humanoid_input.num_frames == len(humanoid_input.domain_names) + 1
print(f"Prepared {row['name']}: {humanoid_input.image.width}×{humanoid_input.image.height} input, "
      f"{humanoid_input.num_frames} frames at {humanoid_input.fps} FPS, seed {humanoid_input.seed}")
print("Canvas the server encodes:", (humanoid_input.width, humanoid_input.height))
print("Expected output size:", (humanoid_input.output_width, humanoid_input.output_height))
print("Domain runs:", domain_runs(humanoid_input.domain_names))

### Preview the inputs

The image is the starting observation. The heatmap shows all 59 normalized action channels over the 30-second sequence, grouped as in the table above. Dashed lines mark every domain switch; divide a run's first row by the FPS to get its time in seconds.

In [ ]:
action_values = humanoid_input.action
groups = [
    (0, 9, "Ego/camera pose"), (9, 18, "Right-wrist pose"),
    (18, 33, "Right fingertips"), (33, 34, "Right gripper"),
    (34, 43, "Left-wrist pose"), (43, 58, "Left fingertips"),
    (58, 59, "Left gripper"),
]
fig, (image_ax, action_ax) = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
image_ax.imshow(humanoid_input.image)
image_ax.set(title="Humanoid: initial observation")
image_ax.axis("off")
limit = max(float(np.abs(action_values).max()), 1e-6)
heatmap = action_ax.imshow(
    action_values.T, aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit,
    extent=(0, len(action_values) / humanoid_input.fps, 58.5, -0.5),
)
action_ax.set_yticks([(start + end - 1) / 2 for start, end, _ in groups],
                    [label for _, _, label in groups])
for _, end, _ in groups[:-1]:
    action_ax.axhline(end - 0.5, color="black", linewidth=0.5)
for _, start, _ in domain_runs(humanoid_input.domain_names)[1:]:
    action_ax.axvline(start / humanoid_input.fps, color="black", linestyle="--", linewidth=1)
action_ax.set(title=f"{row['name']}: {action_values.shape[0]} × {action_values.shape[1]}", xlabel="Time (seconds)")
fig.colorbar(heatmap, ax=action_ax, label="Checkpoint-normalized value", shrink=0.8)
display(fig)
plt.close(fig)

## 5. Generate with `POST /v1/videos`

The request uploads the initial image as `input_reference` and sends the complete action sequence with its per-row domain names in JSON-encoded `extra_params`. The `size` field names the 832×480 canvas: the server pads the 640×360 image at the bottom and right to encode it, then drops the padded latent rows and columns, so the transformer and the output use only the 640×352 content region. Per-row domain names switch from AgiBot to camera and back within this single request. The step count and guidance are fixed by the distilled checkpoint, so the request does not send them. SGLang returns a job ID; the helper polls `GET /v1/videos/{id}` and downloads `/content` when generation completes.

The helper writes the action rows in their shortest float32 form and checks that the payload stays within the server's 1 MiB multipart-field limit; a longer rollout than that limit allows fails before anything is sent. It waits up to `HTTP_TIMEOUT` seconds and saves request and job JSON sidecars beside the MP4. HTTP and generation errors are saved to an `.error.txt` sidecar. Rerunning this cell creates a new job.

In [ ]:
prepared = humanoid_input
output = OUTPUT_DIR / row["name"] / f"{row['name']}.mp4"
started = time.monotonic()
with httpx.Client(base_url=SGLANG_BASE_URL, headers=HEADERS, timeout=HTTP_TIMEOUT) as client:
    output = submit_bimanual(
        client, prepared, ACTIVE_MODEL, output,
        timeout=HTTP_TIMEOUT, poll_interval=5,
    )
elapsed_seconds = time.monotonic() - started
print(f"{row['name']}: {elapsed_seconds:.1f}s — {output}")

## 6. Verify and view the output

Decode the generated video and check that it contains the requested frame count at the requested FPS and the expected output size. For the published image, the output keeps the 640-pixel width; its 360-pixel height becomes 352, the largest multiple of the 16-pixel latent stride, because the padded latent rows are dropped rather than decoded.

Review the rollout against the domain runs printed in step 4: manipulation should follow the `agibotworld` rows and camera motion the `camera_pose` rows, with each switch at its run's first row divided by the FPS. At the release revision the two longest blocks switch at 10 and 20 seconds, after a camera lead-in of under a second, and the final six seconds alternate several times.

In [ ]:
probe = subprocess.run([
    "ffprobe", "-v", "error", "-count_frames", "-select_streams", "v:0",
    "-show_entries", "stream=width,height,avg_frame_rate,nb_read_frames", "-of", "json", str(output),
], check=True, capture_output=True, text=True)
stream = json.loads(probe.stdout)["streams"][0]
assert int(stream["nb_read_frames"]) == prepared.num_frames, stream
assert Fraction(stream["avg_frame_rate"]) == Fraction(str(prepared.fps)), stream
assert (int(stream["width"]), int(stream["height"])) == (prepared.output_width, prepared.output_height), stream
subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-xerror", "-i", str(output), "-f", "null", "-"], check=True)
print(stream)

In [ ]:
display(Video(str(output), embed=True, html_attributes="controls loop"))

## 7. Next steps

- **Run the other published examples:** set `COSMOS3_BIMANUAL_MANIFEST` to `examples/inference/sim_bimanual_unifiedv1_hf.jsonl` and `COSMOS3_BIMANUAL_ACTION` to a sidecar basename from it, such as `agibot_001_normalized_unifiedv1.json`, then restart the notebook.
- **Use your own recording:** set `ACTION_PATH` to a `normalized_unified_v1` sidecar, `VISION_PATH` to the initial image or a video whose first frame is the initial observation, and `HUMANOID_PROMPT` to the task description, then restart the notebook. Keep `num_frames - 1` equal to the number of action rows and divisible by four, and do not normalize the values again. The [Bimanual guide](bimanual.md#download-the-model-and-examples) describes the sidecar contract.
- **Validate further:** apply the [shared Simulation checklist](../README.md#validate-a-rollout). Inspect the complete rollout for frozen frames, discontinuities, and disagreement with the supplied actions, including every domain transition in the printed domain runs. Use a new output directory when changing the checkpoint, inputs, or server settings.
- **Other runtimes:** the [vLLM-Omni notebook](run_bimanual_with_vllm_omni.ipynb) serves the same published example through the synchronous Videos API, and the [Bimanual guide](bimanual.md#run-inference) runs it natively with Cosmos Framework.